# A1 — Gradientno pospešena drevesa: Telekom

Nadaljujemo z istim ciljem in delitvijo kot pri logistični regresiji: ena vrstica je naročnik,
`odhod=1` pomeni odhod. Časovni horizont oznake v viru ni potrjen.


**Razlaga priprave:** [priprava Telekoma](../../../03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov/A1_Priprava_Telekom.ipynb)
in [delitev ter dopolnjevanje](../../../03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela/A1_Delitev_in_manjkajoce_Telekom.ipynb).
Notebook je samostojen: ne potrebuje spremenljivk iz [logistične regresije](../../../03_Uvod_v_strojno_ucenje/notebooks/05_logisticna_regresija/A1_Logisticna_regresija_Telekom.ipynb).

Podatke po potrebi pripravi `poskrbi_za_modeliranje`; `preberi_tabelo` jih bere s tipi iz sheme.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from pathlib import Path
from time import perf_counter
from textwrap import fill
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.special import expit, softmax
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, roc_auc_score, ConfusionMatrixDisplay,
)
try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError("Namestite requirements.txt v okolje tega jedra in jedro ponovno zaženite.") from e

from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
SEME, N_JOBS = 42, 2
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 14)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})
print("LightGBM:", lgb.__version__)

## 1. Isti učni in validacijski podatki
Testne tabele ne odpremo. Na učnih podatkih učimo model in kodiranje; validacija je za primerjavo
nastavitev in zgodnje ustavljanje. Tabel ne spreminjamo in ne ustvarjamo nove delitve.

Začnemo z istimi **izvornimi značilkami in njihovimi indikatorji manjkanja** kot pri logistični regresiji.
`dohodek_razred` ponovno obravnavamo kot kategorijo. Razširjeni nabor preizkusimo pozneje.

In [ ]:
shema = preberi_shemo(PODATKI / "shema.json")
CILJ, KLJUC = shema["cilj"], shema["kljuc"]
train = preberi_tabelo(PODATKI / "train.csv", shema)
val = preberi_tabelo(PODATKI / "val.csv", shema)
assert train[KLJUC].is_unique and val[KLJUC].is_unique
assert set(train[KLJUC]).isdisjoint(val[KLJUC])
VSE = shema["znacilke"]
KAT = shema["kategoricne_znacilke"] + ["dohodek_razred"]
izvorne = shema["nabori_znacilk"]["izvorni"]
indikatorji = [s for s in shema["skupine_znacilk"]["manjkanje"]
              if s.removesuffix("_manjka") in izvorne]
OSNOVNE = izvorne + indikatorji
assert len(OSNOVNE) == len(set(OSNOVNE))
assert not set(shema["metapodatki"] + [CILJ]) & set(VSE)
X_train, X_val = train[VSE].copy(), val[VSE].copy()
y_train, y_val = train[CILJ].copy(), val[CILJ].copy()
for X in [X_train, X_val]:
    assert not X.isna().any().any(), "Uporabite že dopolnjene tabele iz prejšnjega poglavja."
    X[KAT] = X[KAT].astype(str)
    assert np.isfinite(X[[s for s in VSE if s not in KAT]].to_numpy(float)).all()
assert set(y_train.unique()) == set(y_val.unique()) == {0, 1}
display(pd.DataFrame([
    {"zbirka": ime, "vrstic": len(y), "delež odhodov": y.mean()}
    for ime, y in [("učna", y_train), ("validacijska", y_val)]
]).round(4))
print(f"Začetni nabor: {len(OSNOVNE)} značilk; razširjeni: {len(VSE)}.")

### Priprava: kategorije kodiramo, številk ne standardiziramo
Drevesa primerjajo vrednosti s pragovi. Za ta model standardizacije ne potrebujemo.
Ohranimo one-hot kodiranje iz prejšnjega notebooka, da lahko primerjamo iste informacije in beremo razcepe.
LightGBM sicer podpira tudi kategorije neposredno; to je druga možnost priprave.

Mediane so že naučene v prejšnjem poglavju. Tukaj se na učnih podatkih nauči samo slovar kategorij.
Za neznano kategorijo nastanejo same ničle v njeni skupini. To ni isto kot znana oznaka `NEZNANO`.

In [ ]:
def pripravi_znacilke(nabor):
    numericne = [s for s in nabor if s not in KAT]
    kategoricne = [s for s in nabor if s in KAT]
    priprava = ColumnTransformer([
        ("num", "passthrough", numericne),
        ("kat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), kategoricne),
    ], remainder="drop")
    Xt = priprava.fit_transform(X_train[nabor])
    Xv = priprava.transform(X_val[nabor])
    imena = priprava.get_feature_names_out()
    # Povezava kodiranih stolpcev s prvotnimi značilkami za seštevanje pomembnosti.
    izvor = list(numericne)
    if kategoricne:
        for s, kategorije in zip(kategoricne, priprava.named_transformers_["kat"].categories_):
            izvor.extend([s] * len(kategorije))
    assert len(izvor) == len(imena) == Xt.shape[1] == Xv.shape[1]
    assert np.isfinite(Xt).all() and np.isfinite(Xv).all()
    return priprava, Xt, Xv, imena, np.array(izvor)

priprava, Xt, Xv, IMENA, IZVOR = pripravi_znacilke(OSNOVNE)
print("Po kodiranju:", Xt.shape, Xv.shape)
display(pd.DataFrame({"kodiran stolpec": IMENA, "izvorna značilka": IZVOR}).head(8))

## 2. Začetna ocena še ni prvo drevo
Brez značilk vsakemu naročniku napovemo isti **učni delež odhodov**.
Pri binarni logaritemski izgubi ta delež pretvorimo v začetni logit:

**z₀ = ln(p₀ / (1 − p₀))**.

Prvo drevo se nato nauči popravka. V tem notebooku uporabljamo `boost_from_average=True`,
brez uteži razredov in brez posebej podanih začetnih ocen, zato velja spodnji izračun.

In [ ]:
p0 = float(y_train.mean())
z0 = float(np.log(p0 / (1 - p0)))
izh = DummyClassifier(strategy="prior").fit(Xt, y_train)
p_izh = izh.predict_proba(Xv)[:, 1]
np.testing.assert_allclose(p_izh, p0)
np.testing.assert_allclose(expit(z0), p0)
print(f"Začetna verjetnost: {p0:.4f}; začetna ocena z₀: {z0:.4f}")
print(f"Izhodiščni validacijski log loss: {log_loss(y_val, p_izh):.4f}")

## 3. Najprej model s tremi majhnimi drevesi
Za razlago namenoma uporabimo samo **3 drevesa, največ 4 liste in globino 2**.
Ta model ni naš končni kandidat. `learning_rate=0.2` pomeni manjši prispevek vsakega dodanega drevesa.

LightGBM razceplja list, pri katerem pričakuje največje izboljšanje. Drevo zato ni nujno simetrično.
`max_depth` in `num_leaves` omejujeta različni lastnosti drevesa.

In [ ]:
DEMO_PARAMETRI = dict(n_estimators=3, learning_rate=0.2, num_leaves=4, max_depth=2)
majhen = lgb.LGBMClassifier(
    objective="binary", **DEMO_PARAMETRI, min_child_samples=100,
    boost_from_average=True, random_state=SEME, n_jobs=N_JOBS,
    deterministic=True, force_col_wise=True, verbosity=-1,
)
majhen.fit(Xt, y_train)
drevesa_demo = majhen.booster_.dump_model(num_iteration=-1)["tree_info"]
print("Naučenih dreves:", len(drevesa_demo))
display(pd.DataFrame([
    {"drevo": t["tree_index"] + 1, "listov": t["num_leaves"]}
    for t in drevesa_demo
]))

### Kako beremo drevo?
V vozlišču pogledamo značilko in prag. Leva veja pomeni, da pogoj **≤** velja, desna, da ne velja.
Oznake v grafu so zaokrožene; za odločitev uporabimo polno shranjeno vrednost praga.
V listu dobimo številčno vrednost. `n` pove število učnih primerov v vozlišču, `gain` pa izboljšanje pri razcepu.
Pri one-hot stolpcu prag med 0 in 1 v praksi ločuje odsotnost in prisotnost kategorije.

Spodnja pomožna celica samo izriše drevo in prebere pot. Za uporabo je ni treba spreminjati.

In [ ]:
def preberi_pot(koren, vrstica, imena):
    vozlisce, pot = koren, []
    while "split_index" in vozlisce:
        assert vozlisce["decision_type"] == "<=", "Ta prikaz je za numerične/one-hot razcepe."
        j, prag = vozlisce["split_feature"], float(vozlisce["threshold"])
        vrednost = float(vrstica[j])
        assert np.isfinite(vrednost), "V tem prikazu so manjkajoče vrednosti že dopolnjene."
        levo = vrednost <= prag
        pot.append({"vozlišče": vozlisce["split_index"], "značilka": imena[j],
                    "vrednost": vrednost, "prag ≤": prag,
                    "veja": "levo" if levo else "desno"})
        vozlisce = vozlisce["left_child" if levo else "right_child"]
    return pot, vozlisce


def narisi_drevo(koren, imena, pot, list_id, naslov):
    # Pomožni izris; do 16 listov ostane pregleden brez dodatnih programov.
    def listi(n):
        return 1 if "split_index" not in n else listi(n["left_child"]) + listi(n["right_child"])
    if listi(koren) > 16:
        print("Drevo je za pregledni izris preveliko. Uporabite tabelo poti ali zmanjšajte globino.")
        return
    izbrana = {p["vozlišče"] for p in pot}
    fig, ax = plt.subplots(figsize=(max(10, 2.8 * listi(koren)), 6))
    ax.axis("off")
    pozicije, vozlisca, povezave = {}, {}, []
    def postavi(n, globina=0, zac=0):
        kljuc = ("s", n["split_index"]) if "split_index" in n else ("l", n.get("leaf_index", 0))
        if "split_index" in n:
            levo = listi(n["left_child"])
            a = postavi(n["left_child"], globina+1, zac)
            b = postavi(n["right_child"], globina+1, zac+levo)
            x = (pozicije[a][0]+pozicije[b][0])/2
            povezave.extend([(kljuc,a,"DA"),(kljuc,b,"NE")])
        else:
            x = zac
        pozicije[kljuc]=(x,-globina)
        vozlisca[kljuc]=n
        return kljuc
    postavi(koren)
    for a,b,oznaka in povezave:
        x,y=pozicije[a]; xx,yy=pozicije[b]
        ax.plot([x,xx],[y,yy],color="#9aa4ac",zorder=1)
        ax.text((x+xx)/2,(y+yy)/2,oznaka,ha="center",fontsize=10,
                bbox=dict(facecolor="white",edgecolor="none",pad=1))
    for kljuc,n in vozlisca.items():
        if kljuc[0]=="s":
            label=(fill(str(imena[n["split_feature"]]), 27)
                   + f"\n≤ {float(n['threshold']):.6g}\ngain={n['split_gain']:.2f}; n={n['internal_count']}")
            izbran=kljuc[1] in izbrana
        else:
            label=f"List {kljuc[1]}\nvrednost={n['leaf_value']:+.4f}\nn={n['leaf_count']}"
            izbran=kljuc[1]==list_id
        ax.text(*pozicije[kljuc],label,ha="center",va="center",fontsize=10,
                bbox=dict(boxstyle="round,pad=0.45",facecolor="#dceef0" if izbran else "#f2f4f6",
                          edgecolor="#168a87" if izbran else "#9aa4ac"),zorder=2)
    ax.set_xlim(-.65,listi(koren)-.35)
    ax.set_ylim(min(y for x,y in pozicije.values())-.65,.65)
    ax.set_title(naslov,pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# PREIZKUSITE: izberite drugo drevo (od 1 do števila dreves) ali drugo validacijsko vrstico.
DREVO = 1
VRSTICA = 0
assert 1 <= DREVO <= len(drevesa_demo)
assert 0 <= VRSTICA < len(Xv)
koren = drevesa_demo[DREVO - 1]["tree_structure"]
pot, list_ = preberi_pot(koren, Xv[VRSTICA], IMENA)
list_id = list_.get("leaf_index", 0)
listi_modela = np.asarray(majhen.predict(Xv[[VRSTICA]], pred_leaf=True)).reshape(1, -1)
assert list_id == int(listi_modela[0, DREVO - 1])
narisi_drevo(koren, IMENA, pot, list_id, f"Drevo {DREVO} · označena pot validacijskega primera {VRSTICA}")
display(pd.DataFrame(pot).round(4))
print(f"Doseženi list {list_id}; shranjena vrednost: {list_['leaf_value']:+.6f}")

## 4. Kako iz listov dobimo verjetnost?
LightGBM ima pomembno podrobnost zapisa: pri uporabljeni inicializaciji je **začetna ocena vključena v liste prvega drevesa**.
Shranjene listne vrednosti že vključujejo tudi hitrost učenja. Zato začetne ocene ne prištejemo še enkrat in listov
ne množimo ponovno z `learning_rate`.

Za razlago po prosojnici od prve listne vrednosti odštejemo z₀ in tako dobimo ločen **prvi popravek**.
Nato seštejemo z₀ in vse popravke. Na koncu uporabimo sigmoid.

In [ ]:
# Isti naročnik kot zgoraj. Ujemanje preverimo tudi po vsakem dodanem drevesu.
vrstica = Xv[[VRSTICA]]
izbrani_listi = np.asarray(majhen.predict(vrstica, pred_leaf=True)).reshape(1, -1)[0]
shranjene = np.array([
    majhen.booster_.get_leaf_output(i, int(list_id)) for i, list_id in enumerate(izbrani_listi)
])
popravki = shranjene.copy()
popravki[0] -= z0
ocene = z0 + np.cumsum(popravki)
verjetnosti = expit(ocene)
za_prikaz = pd.DataFrame({
    "drevo": np.arange(1, len(shranjene)+1), "doseženi list": izbrani_listi,
    "shranjena vrednost": shranjene, "popravek": popravki,
    "skupaj z": ocene, "verjetnost odhoda": verjetnosti,
})
for k in range(1, len(shranjene)+1):
    np.testing.assert_allclose(ocene[k-1], majhen.predict(vrstica, raw_score=True, num_iteration=k)[0], atol=1e-10)
np.testing.assert_allclose(shranjene.sum(), majhen.predict(vrstica, raw_score=True)[0], atol=1e-10)
np.testing.assert_allclose(verjetnosti[-1], majhen.predict_proba(vrstica)[0,1], atol=1e-12)
print(f"Začetek: z₀={z0:.4f}, p₀={p0:.4f}")
display(za_prikaz.round(5))
print(f"Ročni izračun in model se ujemata: p={verjetnosti[-1]:.5f}.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = ["Začetna ocena"] + [f"Drevo {i}" for i in range(1, len(popravki)+1)]
vrednosti = np.r_[z0, popravki]
axes[0].bar(labels, vrednosti, color=np.where(vrednosti >= 0, "#c45151", "#168a87"))
axes[0].axhline(0, color="grey", lw=1)
axes[0].set(title="Začetna ocena in popravki", ylabel="Prispevek k z")
axes[1].plot(range(len(popravki)+1), np.r_[p0, verjetnosti], marker="o")
axes[1].set(title="Napoved po dodajanju dreves", xlabel="Število dreves", ylabel="Verjetnost odhoda", ylim=(0,1))
axes[1].set_xticks(range(len(popravki)+1))
plt.tight_layout()
plt.show()

**PREIZKUSITE:** izberite drug indeks `VRSTICA` in ponovite pregled poti ter seštevanje.
Ali imajo vsa drevesa isti predznak prispevka? Ali se verjetnost vedno povečuje?

Pri napovedovanju ne učimo pravil in ne računamo gradientov. Vsako drevo bere iste pripravljene značilke;
v njem obiščemo eno pot do lista. Prikaz razloži izračun modela, ne vzrok odhoda naročnika.

## 5. Večji model in zgodnje ustavljanje
Sedaj dovolimo več dreves in več listov. Uporabimo učno ter validacijsko logaritemsko izgubo.
Učenje se ustavi po **40 korakih brez izboljšanja validacijskega log loss** ali ob doseženem proračunu dreves.

Validacijske značilke v `eval_X` morajo biti pripravljene z **istim že naučenim kodiranjem** kot učne.
Zato pripravo izvedemo pred učenjem LightGBM. Navaden `Pipeline` validacijskih podatkov v argumentu `eval_X`
ne pripravi samodejno. Za poznejše napovedovanje oba že naučena koraka povežemo v Pipeline.

In [ ]:
PARAMETRI = dict(
    n_estimators=1000, learning_rate=0.07, num_leaves=15, max_depth=5,
    min_child_samples=100, min_split_gain=0.0,
    subsample=1.0, subsample_freq=0, colsample_bytree=1.0,
    reg_alpha=0.0, reg_lambda=1.0,
)
POTRPEZLJIVOST = 40
# Sem se zbirajo tudi poznejši poskusi. Ponovna izvedba te celice ponastavi zgodovino.
POSKUSI = {}


def meritve(y, p, prag=0.5):
    napoved = np.asarray(p) >= prag
    return {"log loss": log_loss(y, p, labels=[0,1]), "ROC AUC": roc_auc_score(y,p),
            "accuracy": accuracy_score(y,napoved),
            "precision": precision_score(y,napoved,zero_division=0),
            "recall": recall_score(y,napoved,zero_division=0),
            "F1": f1_score(y,napoved,zero_division=0)}


def nauci_poskus(ime, spremembe=None, nabor=None, ustavljanje=True):
    nabor = list(OSNOVNE if nabor is None else nabor)
    if not nabor or len(set(nabor)) != len(nabor) or not set(nabor) <= set(VSE):
        raise ValueError("Nabor mora vsebovati veljavne, enolične vhodne značilke.")
    nastavitve = {**PARAMETRI, **(spremembe or {})}
    if nastavitve["subsample"] < 1 and nastavitve["subsample_freq"] == 0:
        raise ValueError("Za vzorčenje vrstic nastavite tudi subsample_freq > 0.")
    start = perf_counter()
    prep, xtr, xva, imena, izvor = pripravi_znacilke(nabor)
    kandidat = lgb.LGBMClassifier(
        objective="binary", **nastavitve, boost_from_average=True,
        random_state=SEME, n_jobs=N_JOBS, deterministic=True,
        force_col_wise=True, verbosity=-1,
    )
    callbacks = [lgb.log_evaluation(period=0)]
    if ustavljanje:
        callbacks.append(lgb.early_stopping(POTRPEZLJIVOST, first_metric_only=True, verbose=False))
    kandidat.fit(xtr, y_train, eval_X=(xtr,xva), eval_y=(y_train,y_val),
                  eval_names=["ucna","validacijska"], eval_metric="binary_logloss", callbacks=callbacks)
    sekunde = perf_counter()-start
    k = kandidat.best_iteration_ if ustavljanje else kandidat.booster_.current_iteration()
    assert k > 0
    pt = kandidat.predict_proba(xtr, num_iteration=k)[:,1]
    pv = kandidat.predict_proba(xva, num_iteration=k)[:,1]
    zgodovina = kandidat.evals_result_
    if ustavljanje:
        assert k == int(np.argmin(zgodovina["validacijska"]["binary_logloss"]))+1
    np.testing.assert_allclose(log_loss(y_val,pv), zgodovina["validacijska"]["binary_logloss"][k-1], atol=1e-10)
    rezultat = dict(ime=ime, model=kandidat, priprava=prep, nabor=nabor,
        imena=imena, izvor=izvor, iteracija=k, zgodovina=zgodovina,
        p_train=pt, p_val=pv, nastavitve=nastavitve, ustavljanje=ustavljanje,
        povzetek={"poskus":ime, "značilk":len(nabor), "iteracija":k,
                  "izvedenih korakov":len(zgodovina["validacijska"]["binary_logloss"]),
                  "učni log loss":log_loss(y_train,pt), **meritve(y_val,pv), "sekunde učenja":sekunde})
    POSKUSI[ime] = rezultat  # Isti naziv poskusa zamenja njegovo prejšnjo različico.
    return rezultat

osnovni = nauci_poskus("Osnovni model")
model = osnovni["model"]
K = osnovni["iteracija"]
display(pd.DataFrame([osnovni["povzetek"]]).round(4))
koraki = osnovni["povzetek"]["izvedenih korakov"]
print(f"Izbrana iteracija: {K}; izvedenih korakov: {koraki}; proračun: {PARAMETRI['n_estimators']}.")
if koraki >= PARAMETRI["n_estimators"]:
    print("Dosežen je proračun. Daljše učenje bi lahko še izboljšalo validacijo.")

In [ ]:
zgodovina = osnovni["zgodovina"]
fig, ax = plt.subplots()
for kljuc, oznaka in [("ucna","Učna"),("validacijska","Validacijska")]:
    vrednosti = zgodovina[kljuc]["binary_logloss"]
    ax.plot(np.arange(1,len(vrednosti)+1), vrednosti, label=oznaka)
ax.axvline(K, color="#168a87", linestyle="--", label=f"Izbrana iteracija {K}")
ax.set(xlabel="Število dodanih dreves", ylabel="Log loss · manj je bolje", title="Učenje in zgodnje ustavljanje")
ax.legend()
plt.tight_layout()
plt.show()

### Primerjava z izhodiščem in majhnim modelom
`accuracy`, precision, recall in F1 računamo pri pragu 0,5. ROC AUC primerja razvrščanje po oceni prek pragov,
log loss pa kakovost verjetnosti. Precision in F1 izpišemo kot 0, če ni pozitivnih napovedi.
Validacijski rezultat uporabljamo pri razvoju; ni končna neodvisna testna ocena.

In [ ]:
p_majhen = majhen.predict_proba(Xv)[:,1]
primerjava = pd.DataFrame([
    {"model": "Učni delež odhodov", **meritve(y_val,p_izh)},
    {"model": "Tri majhna drevesa", **meritve(y_val,p_majhen)},
    {"model": "Večji model + early stopping", **meritve(y_val,osnovni["p_val"])},
]).set_index("model")
display(primerjava.round(4))
fig, ax = plt.subplots(figsize=(5.5,4.5))
ConfusionMatrixDisplay.from_predictions(y_val, osnovni["p_val"] >= .5,
    labels=[0,1], display_labels=["Brez odhoda","Odhod"], cmap="Blues", colorbar=False, ax=ax)
ax.set(title="Validacija · prag 0,5", xlabel="Napoved", ylabel="Dejanski izid")
plt.tight_layout()
plt.show()

In [ ]:
# Povežemo že naučena koraka. Ne kličemo fit še enkrat.
celotni_postopek = Pipeline([("priprava", osnovni["priprava"]), ("model", model)])
novi = X_val[OSNOVNE].iloc[:5]
p_novi = celotni_postopek.predict_proba(novi, num_iteration=K)[:,1]
np.testing.assert_allclose(p_novi, osnovni["p_val"][:5])
display(pd.DataFrame({"verjetnost odhoda":p_novi, "napoved pri 0,5":p_novi>=.5}).round(4))

## 6. Pomembnost: split count in skupni gain
- **Split count:** koliko razcepov v modelu uporabi značilko.
- **Skupni gain:** vsota izboljšanj učnega kriterija pri teh razcepih.

V LightGBM se drugi način imenuje `gain`. V XGBoostu bi za vsoto uporabili `total_gain`;
tam `gain` pomeni povprečje na razcep. Pomembnost izračunamo za **isto izbrano iteracijo**, kot jo uporabljamo za napovedi.
To niso prispevki k napovedi posameznega naročnika.

In [ ]:
def pomembnosti(poskus):
    booster, k = poskus["model"].booster_, poskus["iteracija"]
    tabela = pd.DataFrame({
        "stolpec": poskus["imena"], "značilka": poskus["izvor"],
        "split count": booster.feature_importance(importance_type="split", iteration=k),
        "skupni gain": booster.feature_importance(importance_type="gain", iteration=k),
    })
    # One-hot kategorije združimo nazaj pod izvorno značilko.
    zdruzeno = tabela.groupby("značilka")[["split count","skupni gain"]].sum()
    zdruzeno["povprečni gain"] = zdruzeno["skupni gain"] / zdruzeno["split count"].replace(0,np.nan)
    total = zdruzeno["skupni gain"].sum()
    zdruzeno["delež skupnega gaina (%)"] = 100*zdruzeno["skupni gain"]/total if total>0 else 0.0
    return tabela, zdruzeno

pom_stolpci, pom_znacilke = pomembnosti(osnovni)
display(pom_znacilke.sort_values("skupni gain",ascending=False).head(12).round(3))
fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax, metrika, barva in zip(axes,["split count","skupni gain"],["#2993ad","#94ad35"]):
    izbrane = pom_znacilke[metrika].nlargest(10).sort_values()
    ax.barh(izbrane.index,izbrane,color=barva)
    ax.set(title=metrika, xlabel=metrika)
plt.tight_layout()
plt.show()

### Preštejmo razcepe in gain tudi sami
Podatke lahko dobimo iz strukture naučenih dreves. S tem preverimo pomen obeh lestvic,
ne da bi model ponovno učili. V tabeli spodaj je vsak razcep ena vrstica.

In [ ]:
def zberi_razcepe(booster, k, imena):
    vrstice = []
    def obisci(n, drevo):
        if "split_index" not in n:
            return
        j = n["split_feature"]
        vrstice.append({"drevo":drevo, "vozlišče":n["split_index"],
                        "indeks značilke":j, "stolpec":imena[j],
                        "prag":n["threshold"], "gain":n["split_gain"], "učnih primerov":n["internal_count"]})
        obisci(n["left_child"],drevo)
        obisci(n["right_child"],drevo)
    for t in booster.dump_model(num_iteration=k)["tree_info"]:
        obisci(t["tree_structure"],t["tree_index"]+1)
    return pd.DataFrame(vrstice,columns=["drevo","vozlišče","indeks značilke","stolpec","prag","gain","učnih primerov"])

razcepi = zberi_razcepe(model.booster_, K, osnovni["imena"])
rocno = razcepi.groupby("indeks značilke").agg(stevilo=("gain","size"), gain=("gain","sum"))
rocno = rocno.reindex(range(len(osnovni["imena"])),fill_value=0)
np.testing.assert_array_equal(rocno.stevilo, pom_stolpci["split count"])
np.testing.assert_allclose(rocno.gain, pom_stolpci["skupni gain"], rtol=1e-6, atol=1e-6)
display(razcepi.head(8).round(3))
print("Ročno prešteti razcepi in sešteti gain se ujemajo s knjižnico.")

**PREIZKUSITE:** izberite eno izvorno značilko in poglejte, kateri kodirani stolpci prispevajo k njeni pomembnosti.
Poiščite značilko, ki je pogosta pri razcepih, a ima manjši skupni gain.

In [ ]:
IZBRANA_ZNACILKA = "regija"
display(pom_stolpci.loc[pom_stolpci["značilka"].eq(IZBRANA_ZNACILKA)]
        .sort_values("skupni gain",ascending=False).head(10).round(3))
uvrstitve = pom_znacilke[["split count","skupni gain"]].rank(ascending=False,method="min")
uvrstitve["razlika uvrstitev"] = uvrstitve["split count"]-uvrstitve["skupni gain"]
display(uvrstitve.loc[uvrstitve["razlika uvrstitev"].abs().nlargest(5).index])

**Omejitve razlage:** pomembnost nima predznaka in ne dokazuje vzročnosti. Povezane značilke se lahko nadomeščajo;
veliko možnih razcepov lahko vpliva na izbor. Pomembnosti iz učnih dreves ne zagotavljajo koristi na novih podatkih.
Delež 30 % skupnega gaina ne pomeni 30 % pravilnih napovedi. Povprečnih gainov kategorij ne seštevamo;
združimo vsoto gaina in število razcepov, nato delimo.

## 7. Hiperparametri: kaj lahko spremenimo?

| Nastavitev | Kaj spreminja? | Predlog za poskus |
| --- | --- | --- |
| `n_estimators` | Zgornja meja števila dreves | 100, 500, 1000 |
| `learning_rate` | Velikost prispevka novega drevesa | 0.03, 0.07, 0.2 |
| `num_leaves` | Največje število listov | 7, 15, 31 |
| `max_depth` | Najdaljša pot od korena do lista | 3, 5, 8 |
| `min_child_samples` | Najmanjša podpora lista (približna omejitev) | 30, 100, 500 |
| `min_split_gain` | Najmanjše izboljšanje za razcep | 0, 1, 5 |
| `subsample` in `subsample_freq` | Vzorčenje vrstic | delež 0.8 in frekvenca 1 |
| `colsample_bytree` | Vzorčenje značilk za drevo | 0.7, 1.0 |
| `reg_alpha`, `reg_lambda` | Kazen L1/L2 za vrednosti listov | 0, 1, 10 |

Najprej spreminjajte eno stvar. Manjša hitrost učenja lahko potrebuje več dreves.
Če `max_depth` omejite na 3, drevo ne more imeti več kot 8 listov, ne glede na višji `num_leaves`.
V LightGBM je omejitev `min_child_samples` izvedena na podlagi ukrivljenosti izgube; dejanski list je lahko manjši.

In [ ]:
# Pripravljena primerjava: isti podatki, spreminjamo eno nastavitev naenkrat.
PRIPRAVLJENI_POSKUSI = {
    "Manj listov": {"num_leaves":7},
    "Več listov": {"num_leaves":31},
    "Manjši korak": {"learning_rate":0.03},
}
for ime, spremembe in PRIPRAVLJENI_POSKUSI.items():
    nauci_poskus(ime, spremembe)


def prikazi_poskuse():
    tabela = pd.DataFrame([p["povzetek"] for p in POSKUSI.values()]).set_index("poskus")
    display(tabela.sort_values("log loss").round(4))
    return tabela

rezultati = prikazi_poskuse()

In [ ]:
fig, ax = plt.subplots()
for ime in ["Osnovni model", *PRIPRAVLJENI_POSKUSI]:
    p = POSKUSI[ime]
    krivulja = p["zgodovina"]["validacijska"]["binary_logloss"]
    ax.plot(np.arange(1,len(krivulja)+1),krivulja,label=ime)
    ax.scatter(p["iteracija"],krivulja[p["iteracija"]-1],s=25)
ax.set(xlabel="Število dreves",ylabel="Validacijski log loss",title="Kako nastavitve spremenijo učenje?")
ax.legend()
plt.tight_layout()
plt.show()

### Vaja: kako štiri omejitve vplivajo na gradnjo dreves?
V spodnji celici prilagajajte **štiri hiperparametre posameznega drevesa**.
Začetne vrednosti so enake kot pri osnovnem modelu. Pred vsakim poskusom napovejte,
kako se bosta spremenili struktura dreves in kakovost napovedi.

| Hiperparameter | Začetna vrednost | Poskus | Kaj opazujemo? |
| --- | --- | --- | --- |
| `max_depth` | 5 | 2 | Kaj izgubimo z omejitvijo globine? |
| `num_leaves` | 15 | 4 | Kako manjše število listov vpliva na napovedi? |
| `min_child_samples` | 100 | 500 | Kaj se zgodi, če za list zahtevamo več primerov? |
| `min_split_gain` | 0.0 | 5.0 | Kaj se zgodi, če zahtevamo večje izboljšanje za razcep? |

**Vsakič obnovite začetne vrednosti in spremenite samo en hiperparameter.**
Poskusu dajte novo ime, da prejšnji ostane v primerjalni tabeli; isto ime prejšnji poskus zamenja.
Ponovno izvedite celico z nastavitvami in naslednjo celico s pregledom dreves.
Nabor značilk, hitrost učenja, največje število dreves in ostale nastavitve ostanejo enaki.
Zgodnje ustavljanje ostane vključeno, zato se lahko število izbranih dreves med poskusi razlikuje.

`max_depth` in `num_leaves` sta **zgornji meji**, ne zahtevani velikosti.
Pri `num_leaves=4` in `max_depth=4` ima drevo največ štiri liste: lahko je uravnoteženo
z globino 2 ali neuravnoteženo z globino 3. Globine 4 ne more doseči, saj bi zanjo
potrebovalo vsaj pet listov. Tudi pri ohlapnejših mejah lahko rast prej ustavita
`min_child_samples` ali `min_split_gain`; omejitev podpore lista je v LightGBM približna.


In [ ]:
IME_POSKUSA = "Vaja: začetne omejitve"  # Za vsak nov poskus izberite novo ime.
NASTAVITVE_DREVES = {
    "max_depth": 5,           
    "num_leaves": 15,         
    "min_child_samples": 100, 
    "min_split_gain": 0.0,
}
assert IME_POSKUSA != osnovni["ime"], "Za primerjavo ohranite osnovni model pod njegovim imenom."
moj = nauci_poskus(IME_POSKUSA, NASTAVITVE_DREVES, ustavljanje=True)
rezultati = prikazi_poskuse()


In [ ]:
# Pregledamo samo drevesa do izbrane iteracije, ki jih model uporablja za napovedi.
def globina_drevesa(vozlisce):
    if "split_index" not in vozlisce:
        return 0  # Drevo z enim listom nima razcepov.
    return 1 + max(globina_drevesa(vozlisce["left_child"]),
                   globina_drevesa(vozlisce["right_child"]))


def povzemi_drevesa(poskus):
    drevesa = poskus["model"].booster_.dump_model(num_iteration=poskus["iteracija"])["tree_info"]
    globine = [globina_drevesa(t["tree_structure"]) for t in drevesa]
    listi = [t["num_leaves"] for t in drevesa]
    return {
        "poskus": poskus["ime"],
        "dreves v napovedi": len(drevesa),
        "povprečna globina": np.mean(globine),
        "največja globina": max(globine),
        "povprečno listov": np.mean(listi),
        "največ listov": max(listi),
    }

struktura_dreves = pd.DataFrame([
    povzemi_drevesa(osnovni), povzemi_drevesa(moj),
]).set_index("poskus")
print("Dejanska struktura naučenih dreves; globino štejemo s številom povezav od korena do najglobljega lista.")
display(struktura_dreves.round(2))


**Razprava in oddaja:** zabeležite nastavitev, največjo globino, povprečno število listov,
učni in validacijski log loss ter spremembo vsaj ene druge metrike glede na osnovni model.

- Ali drevesa dosežejo nastavljene meje? Če jih ne, katera druga omejitev bi jih lahko ustavila?
- Ali preprostejša drevesa poslabšajo učni rezultat? Kaj se zgodi na validaciji?
- Ali se spremeni število dreves, ki jih izbere zgodnje ustavljanje?
- Za zaključek sestavite svojo kombinacijo teh štirih hiperparametrov in utemeljite rezultat.

Nižji log loss je boljši; pri točnosti, natančnosti, priklicu, F1 in ROC AUC je več bolje.
Boljši učni rezultat sam po sebi ne pomeni boljše napovedi za nove naročnike.
Nastavitve primerjamo na validaciji; test ostane zaprt.


### Ali se spremeni tudi pomembnost?
Za isti nabor primerjamo uvrstitvi po skupnem gainu in deleža gaina. Surova vsota gaina med modeli
z različnim številom dreves ni neposredna mera njihove kakovosti.

In [ ]:
_, pom_moj = pomembnosti(moj)
primerjava_pom = pd.DataFrame({
    "osnovni: delež gaina (%)":pom_znacilke["delež skupnega gaina (%)"],
    "moj: delež gaina (%)":pom_moj["delež skupnega gaina (%)"],
    "osnovni: uvrstitev":pom_znacilke["skupni gain"].rank(ascending=False,method="min"),
    "moj: uvrstitev":pom_moj["skupni gain"].rank(ascending=False,method="min"),
})
display(primerjava_pom.sort_values("osnovni: uvrstitev").head(12).round(2))

## 8. Vaja: število avtomobilov in izbor značilk
Ponovimo vprašanje iz logistične regresije: ali število avtomobilov in informacija o njegovem manjkanju pomagata?
Pri vseh treh različicah ohranimo iste hiperparametre in pravilo zgodnjega ustavljanja. Število izbranih dreves se lahko razlikuje.
To je primerjava **ponovno naučenih postopkov**, ne odstranjevanje vhoda iz že naučenega modela.

In [ ]:
AVTO, MANJKA = "stevilo_avtomobilov", "stevilo_avtomobilov_manjka"
print(f"Prvotno manjka število avtomobilov pri {X_train[MANJKA].mean():.1%} učnih vrstic.")
for ime, nabor in {
    "Avtomobili: samo indikator": [s for s in OSNOVNE if s != AVTO],
    "Avtomobili: brez obeh": [s for s in OSNOVNE if s not in [AVTO,MANJKA]],
}.items():
    nauci_poskus(ime,nabor=nabor)
primerjava_avto = pd.DataFrame([POSKUSI[ime]["povzetek"] for ime in
    ["Osnovni model","Avtomobili: samo indikator","Avtomobili: brez obeh"]]).set_index("poskus")
primerjava_avto["razlika log loss glede na osnovni"] = primerjava_avto["log loss"]-primerjava_avto.loc["Osnovni model","log loss"]
display(primerjava_avto[["značilk","iteracija","log loss","recall","precision","razlika log loss glede na osnovni"]].round(5))

### PREIZKUSITE: vse pripravljene značilke ali manjši nabor
Privzeto primerjamo **vseh 64 pripravljenih značilk** z začetnimi 47 pri enakih
hiperparametrih in istem pravilu zgodnjega ustavljanja. Število izbranih dreves se lahko razlikuje.

Možnosti v spodnji celici:
- `"vsi"`: vse značilke iz sheme, tudi pripravljena razmerja, razlike, logaritemske pretvorbe,
  skupina trajanja in dodatni indikatorji manjkanja. ID in cilj nista vhodni značilki.
  `STEVILO_ZNACILK` pri tej izbiri ne omejuje nabora;
- `"top"`: uporabimo `STEVILO_ZNACILK` najvišje uvrščenih značilk iz začetnega nabora
  po skupnem gainu osnovnega modela;
- `"rocno"`: nabor določite sami.

Izpis pokaže število vhodnih značilk pred kodiranjem in njihov seznam. One-hot kodiranje lahko
ustvari več stolpcev, vendar ne dodaja novih izvornih informacij.
Model in kodiranje vsakokrat naučimo ponovno. Primerjamo učni log loss in vse validacijske metrike;
to ni izbor nabora samo na podlagi učne pomembnosti. Pri poznejšem prečnem preverjanju bi tudi
izbor značilk ponavljali znotraj učnih krogov. Test ostane zaprt.

**Za ročni izbor:** spodnji izpis vsebuje vseh 64 imen v obliki Pythonovega seznama.
Kopirajte celoten izpis `ROCNI_NABOR = [...]` na mesto obstoječe vrstice `ROCNI_NABOR`
v celici z nastavitvami, odstranite neželene značilke in nastavite `NACIN_NABORA = "rocno"`.


In [ ]:
# Seznam za kopiranje; vsaka značilka je v svoji vrstici.
print("ROCNI_NABOR = [")
for znacilka in shema["znacilke"]:
    print(f"    {znacilka!r},")
print("]")


In [ ]:
NACIN_NABORA = "vsi"  # "vsi", "top" ali "rocno"
STEVILO_ZNACILK = 12  # Upošteva se samo pri načinu "top".
ROCNI_NABOR = ["mesecev_narocnine", "povp_minute", "povp_mesecni_prihodek", "dni_z_napravo", "regija"]
if NACIN_NABORA == "vsi":
    moj_nabor = list(shema["znacilke"])  # Celoten pripravljeni nabor, ne le OSNOVNE.
elif NACIN_NABORA == "top":
    if not 1 <= STEVILO_ZNACILK <= len(pom_znacilke):
        raise ValueError("Izberite veljavno število značilk.")
    moj_nabor = pom_znacilke["skupni gain"].nlargest(STEVILO_ZNACILK).index.tolist()
elif NACIN_NABORA == "rocno":
    moj_nabor = ROCNI_NABOR
else:
    raise ValueError("Izberite vsi, top ali rocno.")
poskus_nabor = nauci_poskus(f"Nabor {NACIN_NABORA}: {len(moj_nabor)}",nabor=moj_nabor)
assert set(poskus_nabor["izvor"]) == set(moj_nabor), "Kodiranje mora vključiti vse izbrane značilke."
print(f"Uporabljenih vhodnih značilk: {len(moj_nabor)} od {len(VSE)} pripravljenih "
      f"(osnovni model: {len(OSNOVNE)}).")
print("Uporabljene značilke:", moj_nabor)

primerjava_naborov = pd.DataFrame([
    osnovni["povzetek"], poskus_nabor["povzetek"],
]).set_index("poskus")
stolpci_naborov = ["značilk", "iteracija", "učni log loss", "log loss", "ROC AUC",
                   "accuracy", "precision", "recall", "F1"]
print("Log loss: manj je bolje; ostale metrike: več je bolje. Razen učnega log loss so metrike validacijske.")
with pd.option_context("display.max_columns", None):
    display(primerjava_naborov[stolpci_naborov].round(5))
print("Sprememba validacijskega log loss glede na osnovni model:",
      f"{poskus_nabor['povzetek']['log loss'] - osnovni['povzetek']['log loss']:+.5f}")

## 9. Prag spremeni odločitev, ne naučenih dreves
Izberite enega od izvedenih poskusov in spremenite prag. Napovedane verjetnosti ostanejo enake,
spremenita pa se število opozoril in razmerje med precision ter recall.
Praga ne izbiramo na testu. Če ga prilagodimo validaciji, je tudi to del izbire postopka.

In [ ]:
MODEL_ZA_PRAG = "Osnovni model"
MOJ_PRAG = 0.35
assert 0 < MOJ_PRAG < 1
p = POSKUSI[MODEL_ZA_PRAG]["p_val"]
pragi = sorted(set([0.3,0.5,0.7,MOJ_PRAG]))
tabela_pragov = pd.DataFrame([
    {"prag":t, "označenih naročnikov":int((p>=t).sum()), **meritve(y_val,p,prag=t)}
    for t in pragi
]).set_index("prag")
assert tabela_pragov["log loss"].nunique()==1
assert tabela_pragov["ROC AUC"].nunique()==1
display(tabela_pragov.round(4))
fig, ax = plt.subplots(figsize=(5.5,4.5))
ConfusionMatrixDisplay.from_predictions(y_val,p>=MOJ_PRAG,labels=[0,1],
    display_labels=["Brez odhoda","Odhod"],cmap="Blues",colorbar=False,ax=ax)
ax.set(title=f"{MODEL_ZA_PRAG} · prag {MOJ_PRAG}",xlabel="Napoved",ylabel="Dejanski izid")
plt.tight_layout()
plt.show()

## Dodatek A — koliko stane napovedovanje? · neobvezno
Na istem modelu napovedujemo z različno dolgim začetnim delom ansambla. Merimo **samo model** na že kodiranih podatkih,
brez branja datotek in priprave značilk. Pred meritvijo izvedemo ogrevalni klic in nato vzamemo mediano ponovitev.

To je lokalni prikaz, ne produkcijski benchmark. Čas je odvisen tudi od računalnika, števila niti in obremenitve;
kratek model zaradi režijskih stroškov ni nujno sorazmerno hitrejši. Globine dreves tukaj ne spreminjamo.

In [ ]:
# PREIZKUSITE tudi drugo velikost paketa (1, 100, 5000).
VELIKOST_PAKETA = 1000
assert 1 <= VELIKOST_PAKETA <= len(X_val)
x_hitrost = osnovni["priprava"].transform(X_val[OSNOVNE].iloc[:VELIKOST_PAKETA])
vrstice_hitrost=[]
for k in sorted(set([1,min(10,K),min(100,K),K])):
    model.predict_proba(x_hitrost,num_iteration=k)
    casi=[]
    for _ in range(5):
        t=perf_counter()
        model.predict_proba(x_hitrost,num_iteration=k)
        casi.append(perf_counter()-t)
    mediana=float(np.median(casi))
    vrstice_hitrost.append({"dreves":k,"vrstic":VELIKOST_PAKETA,"ms na paket":1000*mediana,
                            "mikrosekund na vrstico v paketu":1e6*mediana/VELIKOST_PAKETA})
display(pd.DataFrame(vrstice_hitrost).round(3))

## Dodatek B — kaj se spremeni pri več razredih? · neobvezno
Za kratek ločen primer uporabimo zbirko **Iris** iz scikit-learn: tri vrste cvetlic, ena pravilna oznaka na vrstico.
To ni nov Telekom cilj. Uporabimo ločeno učno/validacijsko delitev in majhen model.

Model izračuna **tri ocene**. Softmax jih pretvori v verjetnosti, ki se seštejejo v 1.
V običajnem večrazrednem LightGBM vsaka iteracija doda po eno drevo za vsak razred.
Zato 10 iteracij pri treh razredih pomeni 30 dreves. To ni večoznačna naloga, kjer bi lahko veljalo več oznak hkrati.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
iris = load_iris()
Xi_train, Xi_val, yi_train, yi_val = train_test_split(
    iris.data,iris.target,test_size=.3,stratify=iris.target,random_state=SEME,
)
ITERACIJ_IRIS = 10
vecrazredni = lgb.LGBMClassifier(objective="multiclass",n_estimators=ITERACIJ_IRIS,
    num_leaves=4,max_depth=2,min_child_samples=5,learning_rate=.1,
    random_state=SEME,n_jobs=N_JOBS,verbosity=-1,deterministic=True,force_col_wise=True)
vecrazredni.fit(Xi_train,yi_train)
z_vec = vecrazredni.predict(Xi_val[[0]],raw_score=True)[0]
p_vec = vecrazredni.predict_proba(Xi_val[[0]])[0]
np.testing.assert_allclose(softmax(z_vec),p_vec,atol=1e-12)
np.testing.assert_allclose(p_vec.sum(),1.0)
info_vec = vecrazredni.booster_.dump_model()
assert info_vec["num_tree_per_iteration"] == len(vecrazredni.classes_)
assert vecrazredni.booster_.num_trees() == vecrazredni.booster_.current_iteration()*len(vecrazredni.classes_)
display(pd.DataFrame({"razred":iris.target_names[vecrazredni.classes_],"ocena z":z_vec,"softmax(z)":p_vec}).round(4))
print("Iteracij:",vecrazredni.booster_.current_iteration(),"; vseh dreves:",vecrazredni.booster_.num_trees())
print("Dejanski razred prvega primera:",iris.target_names[yi_val[0]])